In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import optuna
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

In [2]:
train = pd.read_csv("TRAIN.csv")
test = pd.read_csv("TEST.csv")

In [3]:
X_full = train.drop("Class", axis=1)
y_full = train["Class"]

model_temp = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_temp.fit(X_full, y_full)

importances = model_temp.feature_importances_
feat_imp = pd.DataFrame({
    "Feature": X_full.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

top_features = feat_imp["Feature"].iloc[:35].tolist()

[LightGBM] [Info] Number of positive: 17311, number of negative: 26465
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 43776, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.395445 -> initscore=-0.424481
[LightGBM] [Info] Start training from score -0.424481


In [4]:
X_final = X_full[top_features]

final_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

final_model.fit(X_final, y_full)

[LightGBM] [Info] Number of positive: 17311, number of negative: 26465
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010807 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8925
[LightGBM] [Info] Number of data points in the train set: 43776, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.395445 -> initscore=-0.424481
[LightGBM] [Info] Start training from score -0.424481


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.03, n_estimators=2000,
               num_leaves=64, random_state=42, subsample=0.8)

In [5]:
X_test_final = test[top_features]

In [6]:
test_probs = final_model.predict_proba(X_test_final)[:,1]

test_preds = (test_probs > 0.405).astype(int)

In [7]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "CLASS": test_preds
})

submission.to_csv("FINAL.csv", index=False)